## 04 -- Table Relationships & Metrics

In [1]:
# Imports and setup

from pathlib import Path 
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

project_root = Path.cwd().parent
raw_dir = project_root / "data" / "raw"

In [2]:
# Initialize Spark Session

spark = (
    SparkSession.builder
    .appName("CLVSourceProfiling")
    .master("local[*]")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/07 13:00:54 WARN Utils: Your hostname, SamDep, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/07 13:00:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/07 13:00:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
order_items = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .csv("../data/raw/order_items.csv")
)

order_items.show(10, truncate=False)

+--------------------------+------------------------+-----------------------+------------------------+------------------------+-------------------+----------+--------+------------------------+-------------+-------------------------------------+----------+-------------+
|APP_NAME                  |RESTAURANT_ID           |CREATION_TIME_UTC      |ORDER_ID                |USER_ID                 |PRINTED_CARD_NUMBER|IS_LOYALTY|CURRENCY|LINEITEM_ID             |ITEM_CATEGORY|ITEM_NAME                            |ITEM_PRICE|ITEM_QUANTITY|
+--------------------------+------------------------+-----------------------+------------------------+------------------------+-------------------+----------+--------+------------------------+-------------+-------------------------------------+----------+-------------+
|Alltown Fresh             |63bc98a7519adc105105a990|2023-03-08 06:03:32.223|64086b8463905bf310002079|63ee144950286a8367041911|NULL               |false     |USD     |64086b84d5fbd8718a0717e

In [4]:
order_options = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .csv(str(raw_dir / "order_item_options.csv"))
)

order_options.show(10, truncate=False)

+------------------------+------------------------+-----------------+---------------+------------+---------------+
|ORDER_ID                |LINEITEM_ID             |OPTION_GROUP_NAME|OPTION_NAME    |OPTION_PRICE|OPTION_QUANTITY|
+------------------------+------------------------+-----------------+---------------+------------+---------------+
|6278eb58dfafab1e053fe172|6278eb63dfafab1e053fe18c|Cheese           |American Cheese|0.0         |1              |
|6278eb58dfafab1e053fe172|6278eb63dfafab1e053fe18c|Veggies          |Lettuce        |0.5         |1              |
|6278eb58dfafab1e053fe172|6278eb63dfafab1e053fe18c|Veggies          |Tomatoes       |0.0         |1              |
|6278eb58dfafab1e053fe172|6278eb63dfafab1e053fe18c|Veggies          |Onion          |0.0         |1              |
|6278eb58dfafab1e053fe172|6278eb63dfafab1e053fe18c|Veggies          |Green Peppers  |0.0         |1              |
|627b87809f91930b3a21b7fa|627b90208270026f7677bfc6|Cheese           |American Ch

In [5]:
# Sum selected option prices at the order-line level

options_by_line = (
    order_options
    .groupBy("ORDER_ID", "LINEITEM_ID")
    .agg(
        F.sum(
            F.col("OPTION_PRICE") * F.col("OPTION_QUANTITY")
        ).alias("options_total"),
        F.count("*").alias("option_record_count")
    )
)

# Preserve every purchased line item, including those without options

items_with_options = (
    order_items
    .join(
        options_by_line,
        on=["ORDER_ID", "LINEITEM_ID"],
        how="left"
    )
    .withColumn(
        "options_total",
        F.coalesce(F.col("options_total"), F.lit(0.0))
    )
    .withColumn(
        "option_record_count",
        F.coalesce(F.col("option_record_count"), F.lit(0))
    )
)

items_with_options.select(
    "RESTAURANT_ID",
    "ITEM_NAME",
    "ITEM_QUANTITY",
    "ITEM_PRICE",
    "options_total",
    "option_record_count"
).show(20, truncate=False)

+------------------------+-------------------------------------+-------------+----------+-------------+-------------------+
|RESTAURANT_ID           |ITEM_NAME                            |ITEM_QUANTITY|ITEM_PRICE|options_total|option_record_count|
+------------------------+-------------------------------------+-------------+----------+-------------+-------------------+
|5e7e35ec902ad5ac017b242a|Breakfast Burrito*                   |1            |5.99      |0.0          |1                  |
|5f6a6c1537ab46bd38e9df75|Greek Salad                          |1            |11.99     |2.0          |2                  |
|622107b40ac81503e0369ca6|Breakfast Burrito*                   |1            |5.99      |0.0          |0                  |
|5e7e35ef902ad5ac017b242c|Power                                |1            |7.0       |0.0          |1                  |
|5f6a6c1537ab46bd38e9df75|Breakfast Burrito*                   |1            |7.99      |2.0          |3                  |
|5f6a6c1

In [6]:
price_comparison = (
    items_with_options
    .filter(
        (F.col("ITEM_QUANTITY") == 1)
        & F.col("ITEM_NAME").isNotNull()
    )
    .groupBy(
        "RESTAURANT_ID",
        "ITEM_NAME",
        "options_total",
        "ITEM_PRICE"
    )
    .agg(F.count("*").alias("transaction_count"))
    .orderBy(
        "RESTAURANT_ID",
        "ITEM_NAME",
        "options_total",
        "ITEM_PRICE"
    )
)

price_comparison.show(50, truncate=False)

+------------------------+------------------------------+-------------+----------+-----------------+
|RESTAURANT_ID           |ITEM_NAME                     |options_total|ITEM_PRICE|transaction_count|
+------------------------+------------------------------+-------------+----------+-----------------+
|5e7e35ec902ad5ac017b242a|ALLTOWN FRESH HOT COFFEE      |0.0          |1.69      |57               |
|5e7e35ec902ad5ac017b242a|ALLTOWN FRESH HOT COFFEE      |0.0          |1.99      |63               |
|5e7e35ec902ad5ac017b242a|ALLTOWN FRESH HOT COFFEE      |0.0          |2.29      |77               |
|5e7e35ec902ad5ac017b242a|ALLTOWN FRESH HOT COFFEE      |0.0          |2.39      |7                |
|5e7e35ec902ad5ac017b242a|ALLTOWN FRESH HOT COFFEE      |0.0          |2.49      |40               |
|5e7e35ec902ad5ac017b242a|ALLTOWN FRESH HOT COFFEE      |0.0          |2.69      |96               |
|5e7e35ec902ad5ac017b242a|ALLTOWN FRESH HOT COFFEE      |0.0          |2.79      |82       

In [7]:
# Compare observed prices for quantity 1 and quantity 2
qty_1 = (
    items_with_options
    .filter(
        (F.col("ITEM_QUANTITY") == 1)
        & (F.col("options_total") == 0)
        & F.col("ITEM_NAME").isNotNull()
    )
    .select(
        "RESTAURANT_ID",
        "ITEM_NAME",
        F.round("ITEM_PRICE", 2).alias("price_qty_1")
    )
    .distinct()
)

qty_2 = (
    items_with_options
    .filter(
        (F.col("ITEM_QUANTITY") == 2)
        & (F.col("options_total") == 0)
        & F.col("ITEM_NAME").isNotNull()
    )
    .select(
        "RESTAURANT_ID",
        "ITEM_NAME",
        F.round("ITEM_PRICE", 2).alias("price_qty_2")
    )
    .distinct()
)

quantity_matches = (
    qty_1
    .join(qty_2, on=["RESTAURANT_ID", "ITEM_NAME"], how="inner")
    .filter(
        F.round(F.col("price_qty_1") * 2, 2)
        == F.col("price_qty_2")
    )
    .orderBy("ITEM_NAME", "RESTAURANT_ID")
)

quantity_matches.show(30, truncate=False)

+------------------------+------------------------+-----------+-----------+
|RESTAURANT_ID           |ITEM_NAME               |price_qty_1|price_qty_2|
+------------------------+------------------------+-----------+-----------+
|61bccd6a7b92a75dc16baf4e|1/2 Garden Salad        |3.99       |7.98       |
|5e7e35ec902ad5ac017b242a|ALLTOWN FRESH HOT COFFEE|1.99       |3.98       |
|5e7e35ec902ad5ac017b242a|ALLTOWN FRESH HOT COFFEE|2.29       |4.58       |
|5e7e35ec902ad5ac017b242a|ALLTOWN FRESH HOT COFFEE|2.69       |5.38       |
|5e7e35ec902ad5ac017b242a|ALLTOWN FRESH HOT COFFEE|1.69       |3.38       |
|5e7e35ec902ad5ac017b242a|ALLTOWN FRESH HOT COFFEE|2.39       |4.78       |
|5e7e35ec902ad5ac017b242a|ALLTOWN FRESH HOT COFFEE|2.49       |4.98       |
|5e7e35ec902ad5ac017b242a|ALLTOWN FRESH HOT COFFEE|2.79       |5.58       |
|5e7e35ed902ad5ac017b242b|ALLTOWN FRESH HOT COFFEE|1.69       |3.38       |
|5e7e35ed902ad5ac017b242b|ALLTOWN FRESH HOT COFFEE|2.49       |4.98       |
|5e7e35ed902

Generate missing calendar dates to align with the transactions table range

In [8]:
order_items.printSchema()

root
 |-- APP_NAME: string (nullable = true)
 |-- RESTAURANT_ID: string (nullable = true)
 |-- CREATION_TIME_UTC: timestamp (nullable = true)
 |-- ORDER_ID: string (nullable = true)
 |-- USER_ID: string (nullable = true)
 |-- PRINTED_CARD_NUMBER: long (nullable = true)
 |-- IS_LOYALTY: boolean (nullable = true)
 |-- CURRENCY: string (nullable = true)
 |-- LINEITEM_ID: string (nullable = true)
 |-- ITEM_CATEGORY: string (nullable = true)
 |-- ITEM_NAME: string (nullable = true)
 |-- ITEM_PRICE: double (nullable = true)
 |-- ITEM_QUANTITY: integer (nullable = true)



In [9]:
order_items.select("CREATION_TIME_UTC").show(truncate=False)

+-----------------------+
|CREATION_TIME_UTC      |
+-----------------------+
|2023-03-08 06:03:32.223|
|2023-05-13 08:45:00.475|
|2023-02-03 02:15:28.519|
|2023-08-17 12:18:02.231|
|2023-08-20 12:02:12.518|
|2023-11-14 12:06:57.484|
|2023-04-15 08:45:16.9  |
|2023-08-15 11:21:34.821|
|2022-05-19 10:03:45.729|
|2023-08-09 11:26:57.377|
|2023-08-09 11:26:57.377|
|2023-02-13 06:38:21.893|
|2023-03-22 13:25:28.805|
|2023-07-24 12:57:26.409|
|2023-08-02 11:03:28.964|
|2023-05-02 12:14:19.93 |
|2023-02-16 07:39:09.526|
|2023-06-14 07:55:21.05 |
|2023-06-14 07:55:21.05 |
|2023-06-26 13:14:04.945|
+-----------------------+
only showing top 20 rows


In [10]:
order_items.select(
    "CREATION_TIME_UTC"
).filter(
    ~F.col("CREATION_TIME_UTC").rlike(r"^\d{4}-\d{2}-\d{2}T")
).show(20, truncate=False)

+-----------------------+
|CREATION_TIME_UTC      |
+-----------------------+
|2023-03-08 06:03:32.223|
|2023-05-13 08:45:00.475|
|2023-02-03 02:15:28.519|
|2023-08-17 12:18:02.231|
|2023-08-20 12:02:12.518|
|2023-11-14 12:06:57.484|
|2023-04-15 08:45:16.9  |
|2023-08-15 11:21:34.821|
|2022-05-19 10:03:45.729|
|2023-08-09 11:26:57.377|
|2023-08-09 11:26:57.377|
|2023-02-13 06:38:21.893|
|2023-03-22 13:25:28.805|
|2023-07-24 12:57:26.409|
|2023-08-02 11:03:28.964|
|2023-05-02 12:14:19.93 |
|2023-02-16 07:39:09.526|
|2023-06-14 07:55:21.05 |
|2023-06-14 07:55:21.05 |
|2023-06-26 13:14:04.945|
+-----------------------+
only showing top 20 rows


In [11]:
# establish the date range from the transaction data min and max

transaction_dates = (
    order_items
    .filter(
        F.col("CREATION_TIME_UTC").rlike(r"^\d{4}-\d{2}-\d{2}T")
    )
    .withColumn(
        "transaction_date",
        F.to_date(F.substring("CREATION_TIME_UTC", 1, 10), "yyyy-MM-dd")
    )
    .agg(
        F.min("transaction_date").alias("start_date"),
        F.max("transaction_date").alias("end_date")
    )
)

# generate one row for every date in that range.
extended_calendar = (
    transaction_dates
    .select(
        F.explode(
            F.sequence(
                F.col("start_date"),
                F.col("end_date"),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("date_key")
    )
)

extended_calendar.show(5)

+--------+
|date_key|
+--------+
+--------+



In [12]:
extended_calendar = (
    extended_calendar
    .withColumn("year", F.year("date_key"))
    .withColumn("month", F.month("date_key"))
    .withColumn("week", F.weekofyear("date_key"))
    .withColumn("day_of_week", F.date_format("date_key", "EEEE"))
    .withColumn(
        "is_weekend",
        F.dayofweek("date_key").isin(1, 7)
    )
)

extended_calendar.show(10, truncate=False)

+--------+----+-----+----+-----------+----------+
|date_key|year|month|week|day_of_week|is_weekend|
+--------+----+-----+----+-----------+----------+
+--------+----+-----+----+-----------+----------+



##### Customer ID Recovery Assessment

Assess whether `PRINTED_CARD_NUMBER` can be used to recover a customer identifier when `USER_ID` is missing.

In [13]:
missing_user = order_items.filter(F.col("USER_ID").isNull())

missing_user.select(
    F.count("*").alias("rows_missing_user_id"),
    F.count("PRINTED_CARD_NUMBER").alias("rows_with_card_number"),
    F.countDistinct("PRINTED_CARD_NUMBER").alias("distinct_card_numbers")
).show()

+--------------------+---------------------+---------------------+
|rows_missing_user_id|rows_with_card_number|distinct_card_numbers|
+--------------------+---------------------+---------------------+
|               17808|                   28|                   18|
+--------------------+---------------------+---------------------+



26/10/07 16:36:13 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1927251 ms exceeds timeout 120000 ms
26/10/07 16:36:16 WARN SparkContext: Killing executors is not supported by current scheduler.
26/10/07 16:36:18 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:85)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:70